# FLEX Replication Notebook

Replication code for **"What's Missing Matters: Negative Feature Attribution for Explainable Classification"**.

This notebook reproduces the main experimental results using **FLEX** (Fisher LDA EXplainer):

1. [FLEX implementation](#1-flex-implementation) — self-contained, numpy only
2. [Evaluation helpers](#2-evaluation-helpers)
3. [Tabular benchmarks](#3-tabular-benchmarks) — USArrests, Iris, Wine, Wisconsin BC, Digits, Mice Protein, Anuran, Avila
4. [Large-scale benchmarks](#4-large-scale-benchmarks) — 20 Newsgroups, CIFAR-10, Covtype *(commented out — long runtime)*
5. [Non-convex boundaries](#5-non-convex-boundaries) — Two Moons, Concentric Circles
6. [Signed attribution experiment](#6-signed-attribution-experiment) — synthetic exclusion dataset

---
**Dependencies:** `numpy scipy scikit-learn pandas ucimlrepo`
```
pip install numpy scipy scikit-learn pandas ucimlrepo
```

## 1. FLEX Implementation

FLEX computes a per-class weight matrix $W = (S_w + \lambda I)^{-1} C^\top$ where $S_w$ is the pooled within-class covariance and $C$ is the centroid matrix. Attribution is via the LDA decision rule $\hat{\ell}(x) = \arg\max_c (x^\top W_{:,c} + b_c)$.

In [ ]:
import numpy as np
import pandas as pd
from scipy.linalg import cho_factor, cho_solve
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')


class FLEX:
    """Fisher LDA EXplainer (Algorithm 1 in the paper)."""

    def __init__(self, lambda_reg=None):
        self.lambda_reg = lambda_reg

    def fit(self, X: np.ndarray, labels: np.ndarray):
        n, d = X.shape
        classes = np.unique(labels)
        k = len(classes)

        # --- Step 1: Compute per-class centroids mu_c ---
        centroids = np.zeros((k, d))
        for i, c in enumerate(classes):
            centroids[i] = X[labels == c].mean(axis=0)

        # --- Step 2: Assemble centroid matrix C (k x d) ---
        C = centroids   # rows are mu_1 ... mu_k

        # --- Step 3: Compute pooled within-class covariance S_w ---
        S_w = np.zeros((d, d))
        for i, c in enumerate(classes):
            diff = X[labels == c] - centroids[i]
            S_w += diff.T @ diff
        S_w /= n

        # --- Step 4: Set lambda = max(median(diag(S_w)), 1e-6) ---
        lam = self.lambda_reg if self.lambda_reg is not None \
            else float(max(np.median(np.diag(S_w)), 1e-6))

        # --- Step 5: Cholesky factorization of S_w + lambda*I = L L^T ---
        c_fac = cho_factor(S_w + lam * np.eye(d))

        # --- Step 6: Cholesky solve for W = (S_w + lambda*I)^{-1} C^T ---
        # W has shape (d, k); column W[:,c] is the weight vector for class c
        self.W_ = cho_solve(c_fac, C.T)
        self.bias_ = -0.5 * np.einsum('cd,dc->c', C, self.W_)
        self.classes_ = classes
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        scores = X @ self.W_ + self.bias_
        return self.classes_[np.argmax(scores, axis=1)]

    def attributions(self, class_idx: int, feature_names=None):
        """Return (positive, negative) attribution lists for one class.

        Positive: features where W[i, class_idx] > 0
        Negative: features where W[i, class_idx] < 0
        """
        w = self.W_[:, class_idx]
        names = (feature_names if feature_names is not None
                 else np.arange(len(w)).astype(str))
        pos = [names[i] for i in np.where(w > 0)[0]]
        neg = [names[i] for i in np.where(w < 0)[0]]
        return pos, neg

## 2. Evaluation Helpers

In [ ]:
from sklearn.metrics import precision_score, recall_score


def evaluate_flex_kmeans(X, n_clusters, n_seeds=10, scale=True):
    """Scale X, fit k-means, fit FLEX on the same scaled data; report macro P and R.

    K-means and FLEX must both operate on the same scaled feature matrix.
    Running k-means on raw X but FLEX on scaled X gives inconsistent clusters.
    """
    ps, rs = [], []
    Xs = StandardScaler().fit_transform(X) if scale else np.array(X, dtype=float)
    for seed in range(n_seeds):
        km = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
        cluster_labels = km.fit_predict(Xs)
        flex = FLEX().fit(Xs, cluster_labels)
        pred = flex.predict(Xs)
        ps.append(precision_score(cluster_labels, pred,
                                  average='macro', zero_division=0))
        rs.append(recall_score(cluster_labels, pred,
                               average='macro', zero_division=0))
    return float(np.mean(ps)), float(np.std(ps)), float(np.mean(rs)), float(np.std(rs))


def print_result(name, p_mean, p_std, r_mean, r_std):
    print(f"{name:<40}  P={p_mean:.3f}\u00b1{p_std:.3f}  R={r_mean:.3f}\u00b1{r_std:.3f}")


## 3. Tabular Benchmarks

Reproduces Table 1 from the paper (small to medium datasets).
Expected results: FLEX achieves P ≥ 0.95 on all datasets except Circles.

In [ ]:
from sklearn.datasets import (load_iris, load_wine, load_breast_cancer,
                               load_digits)

# All datasets: k-means defines the reference partition (paper Table 1 protocol)
print(f"{'Dataset':<40}  {'Macro P':>12}  {'Macro R':>10}")
print("-" * 68)


In [ ]:
# USArrests (k=3) — paper Table 1 uses k=3
url = ("https://raw.githubusercontent.com/vincentarelbundock/"
       "Rdatasets/master/csv/datasets/USArrests.csv")
arrests = pd.read_csv(url, index_col=0)
arrests.columns = ["Murder", "Assault", "UrbanPop", "Rape"]
p, ps, r, rs = evaluate_flex_kmeans(arrests.values, n_clusters=3)
print_result("USArrests (n=50, d=4, k=3)", p, ps, r, rs)


In [ ]:
# Iris (k=3) — k-means partition, not ground-truth labels
data = load_iris()
p, ps, r, rs = evaluate_flex_kmeans(data.data, n_clusters=3)
print_result("Iris (n=150, d=4, k=3)", p, ps, r, rs)


In [ ]:
# Wine (k=3) — k-means partition
data = load_wine()
p, ps, r, rs = evaluate_flex_kmeans(data.data, n_clusters=3)
print_result("Wine (n=178, d=13, k=3)", p, ps, r, rs)


In [ ]:
# Wisconsin BC (k=2) — k-means partition
data = load_breast_cancer()
p, ps, r, rs = evaluate_flex_kmeans(data.data, n_clusters=2)
print_result("Wisconsin BC (n=569, d=30, k=2)", p, ps, r, rs)


In [ ]:
# Digits (k=10) — k-means partition
data = load_digits()
p, ps, r, rs = evaluate_flex_kmeans(data.data, n_clusters=10)
print_result("Digits (n=1797, d=64, k=10)", p, ps, r, rs)


In [ ]:
!pip install -q ucimlrepo

# Mice Protein Expression (k=8)
# The UCI dataset includes categorical meta-columns (Genotype, Treatment,
# Behavior); select numeric protein-expression columns only.
from ucimlrepo import fetch_ucirepo
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

mice = fetch_ucirepo(id=342)
mice_features = pd.DataFrame(mice.data.features).select_dtypes(include='number')
X_mice = SimpleImputer().fit_transform(mice_features.values)
y_mice = LabelEncoder().fit_transform(mice.data.targets.values.ravel())
p, ps, r, rs = evaluate_flex_kmeans(X_mice, n_clusters=len(set(y_mice)))
print_result("Mice Protein (n=1080, d=77, k=8)", p, ps, r, rs)


In [ ]:
# Anuran Calls (MFCCs) — direct UCI download (not available via ucimlrepo API)
# k=10 (Species-level labels); 22 MFCC features
import io, zipfile, urllib.request

anuran_url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
              "00406/Anuran%20Calls%20(MFCCs).zip")
_data = urllib.request.urlopen(anuran_url).read()
with zipfile.ZipFile(io.BytesIO(_data)) as _z:
    anuran_df = pd.read_csv(_z.open('Frogs_MFCCs.csv'))

X_an = anuran_df.iloc[:, :22].values.astype(float)
y_an = LabelEncoder().fit_transform(anuran_df['Species'].values)
p, ps, r, rs = evaluate_flex_kmeans(X_an, n_clusters=len(set(y_an)))
print_result("Anuran (n=7195, d=22, k=10)", p, ps, r, rs)

In [ ]:
# Avila — direct UCI download (not available via ucimlrepo API)
# Combines train (avila-tr.txt) and test (avila-ts.txt) splits; k=12 classes
avila_url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
             "00459/avila.zip")
_data = urllib.request.urlopen(avila_url).read()
_cols = [f'f{i}' for i in range(10)] + ['class']
with zipfile.ZipFile(io.BytesIO(_data)) as _z:
    _tr = pd.read_csv(_z.open('avila/avila-tr.txt'), header=None, names=_cols)
    _ts = pd.read_csv(_z.open('avila/avila-ts.txt'), header=None, names=_cols)
avila_df = pd.concat([_tr, _ts], ignore_index=True)

X_av = avila_df.iloc[:, :10].values.astype(float)
y_av = LabelEncoder().fit_transform(avila_df['class'].values)
p, ps, r, rs = evaluate_flex_kmeans(X_av, n_clusters=len(set(y_av)))
print_result("Avila (n=20867, d=10, k=12)", p, ps, r, rs)

## 4. Large-Scale Benchmarks

These reproduce the remaining rows of Table 1. Commented out by default due to
long download and/or runtime (20 Newsgroups ~2 min, CIFAR-10 ~15 min,
Covtype ~5 min).

In [ ]:
# # 20 Newsgroups (d=1000, k=20)
# # Top-1000 TF-IDF features; sublinear_tf=True, min_df=5, English stop-words.
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.datasets import fetch_20newsgroups
#
# raw = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
# vec = TfidfVectorizer(max_features=1000, sublinear_tf=True,
#                       min_df=5, stop_words='english')
# X_news = vec.fit_transform(raw.data).toarray()
# p, ps, r, rs = evaluate_flex_kmeans(X_news, n_clusters=20)
# print_result("20 Newsgroups (n=18846, d=1000, k=20)", p, ps, r, rs)

In [ ]:
# # CIFAR-10 (d=3072, k=10) — geometric stress test, not semantic clustering
# # Requires tensorflow or keras for dataset download.
# import tensorflow as tf
# (X_tr, y_tr), (X_te, y_te) = tf.keras.datasets.cifar10.load_data()
# X_cifar = np.concatenate([X_tr, X_te]).reshape(-1, 3072).astype(float)
# y_cifar = np.concatenate([y_tr, y_te]).ravel()
# p, ps, r, rs = evaluate_flex_kmeans(X_cifar, n_clusters=10, n_seeds=3)
# print_result("CIFAR-10 (n=60000, d=3072, k=10)", p, ps, r, rs)

In [ ]:
# # Covtype (n=581012, d=54, k=7)
# from sklearn.datasets import fetch_covtype
# cov = fetch_covtype()
# p, ps, r, rs = evaluate_flex_kmeans(cov.data, n_clusters=7, n_seeds=5)
# print_result("Covtype (n=581012, d=54, k=7)", p, ps, r, rs)

## 5. Non-Convex Boundaries

Two Moons uses spectral clustering (RBF kernel) as the reference partition, since
k-means cannot separate the two crescents. FLEX is fitted on degree-2 polynomial
features (d=5) of the raw 2D coordinates and evaluated against the spectral labels.
ExKMC runs its own k-means internally, so its fidelity against the spectral reference
is highly variable depending on whether k-means happens to align with the spectral partition.

Expected: Two Moons FLEX P≈0.960, ExKMC P≈0.573 (high variance).
Circles P≈0.500 for both (shared centroid at origin — no linear signal).

In [ ]:
from sklearn.datasets import make_moons, make_circles
from sklearn.preprocessing import PolynomialFeatures
from sklearn.cluster import SpectralClustering

print(f"{'Dataset':<45}  {'Macro P':>12}  {'Macro R':>10}")
print("-" * 73)

# Two Moons — spectral clustering reference, poly-2 features (d=5)
flex_ps, flex_rs = [], []
for seed in range(10):
    X_m, _ = make_moons(n_samples=1000, noise=0.2, random_state=seed)
    sc = SpectralClustering(n_clusters=2, random_state=seed, affinity='rbf', n_init=10)
    spectral_lbl = sc.fit_predict(X_m)
    X_poly = PolynomialFeatures(2, include_bias=False).fit_transform(X_m)
    X_poly = StandardScaler().fit_transform(X_poly)   # d=5
    flex = FLEX().fit(X_poly, spectral_lbl)
    pred = flex.predict(X_poly)
    flex_ps.append(precision_score(spectral_lbl, pred, average='macro', zero_division=0))
    flex_rs.append(recall_score(spectral_lbl, pred, average='macro', zero_division=0))
print_result("Two Moons FLEX (k=2, d=5, poly-2, spectral ref)",
             np.mean(flex_ps), np.std(flex_ps),
             np.mean(flex_rs), np.std(flex_rs))

# Concentric Circles — raw 2D, shared centroid: both methods reduce to chance
ps_all, rs_all = [], []
for seed in range(10):
    X_c, y_c = make_circles(n_samples=500, noise=0.05, random_state=seed)
    X_c = StandardScaler().fit_transform(X_c)
    flex = FLEX().fit(X_c, y_c)
    pred = flex.predict(X_c)
    ps_all.append(precision_score(y_c, pred, average='macro', zero_division=0))
    rs_all.append(recall_score(y_c, pred, average='macro', zero_division=0))
print_result("Concentric Circles (k=2, d=2)",
             np.mean(ps_all), np.std(ps_all),
             np.mean(rs_all), np.std(rs_all))

## 6. Signed Attribution Experiment

Synthetic 3-cluster dataset where Cluster 0 is defined by the *absence* of
feature `f_excl`. FLEX identifies `f_excl` as a **negative** attribution
for Cluster 0.

**Dataset construction:**
- $n=500$, $d=14$ features, $k=3$ clusters
- `f_excl`: mean $-\delta$ in Cluster 0, mean $+\delta$ in Clusters 1 & 2
- `p1`, `p2`: proxy features separating Clusters 1 and 2
- 10 noise features

In [ ]:
def make_exclusion_dataset(n=500, delta=2.0, seed=0):
    rng = np.random.default_rng(seed)
    n_per = n // 3

    # Cluster 0: f_excl absent (negative), p1/p2 noise
    C0 = rng.standard_normal((n_per, 14))
    C0[:, 0] = rng.normal(-delta, 0.5, n_per)   # f_excl absent
    C0[:, 1] = rng.normal(0, 0.3, n_per)          # p1 neutral
    C0[:, 2] = rng.normal(0, 0.3, n_per)          # p2 neutral

    # Cluster 1: f_excl present, p1 high
    C1 = rng.standard_normal((n_per, 14))
    C1[:, 0] = rng.normal(+delta, 0.5, n_per)   # f_excl present
    C1[:, 1] = rng.normal(+delta, 0.5, n_per)    # p1 high
    C1[:, 2] = rng.normal(0, 0.3, n_per)          # p2 neutral

    # Cluster 2: f_excl present, p2 high
    C2 = rng.standard_normal((n_per, 14))
    C2[:, 0] = rng.normal(+delta, 0.5, n_per)   # f_excl present
    C2[:, 1] = rng.normal(0, 0.3, n_per)          # p1 neutral
    C2[:, 2] = rng.normal(+delta, 0.5, n_per)    # p2 high

    X = np.vstack([C0, C1, C2])
    y = np.array([0]*n_per + [1]*n_per + [2]*n_per)
    names = ['f_excl', 'p1', 'p2'] + [f'noise_{i}' for i in range(11)]
    return X, y, names


X_syn, y_syn, feat_names = make_exclusion_dataset()
X_syn_s = StandardScaler().fit_transform(X_syn)
flex_syn = FLEX().fit(X_syn_s, y_syn)

print("Signed attribution results on synthetic exclusion dataset")
print("=" * 55)
for i, cls in enumerate(flex_syn.classes_):
    pos, neg = flex_syn.attributions(i, feat_names)
    print(f"\nCluster {cls}:")
    print(f"  Positive (present): {pos[:5]}")
    print(f"  Negative (absent):  {neg[:5]}")

pred_syn = flex_syn.predict(X_syn_s)
p_syn = precision_score(y_syn, pred_syn, average='macro', zero_division=0)
r_syn = recall_score(y_syn, pred_syn, average='macro', zero_division=0)
print(f"\nFidelity  P={p_syn:.3f}  R={r_syn:.3f}")
print("\nExpected: Cluster 0 shows f_excl as a NEGATIVE attribution.")